# 11 — RQ1 đánh giá cuối: LLM few-shot trên các tập khóa (máy local)

**Hai phần, hai thời điểm**

Phần A chạy TRƯỚC notebook 10: xuất `review_id` và `text` của Hold-out, focused set và Lazada để tải lên Kaggle. Phần B chạy SAU khi
đã có dự đoán của 10a: gọi LLM với prompt v1 đã khóa.

**Không train, không tune**

Chiến lược C dùng đúng prompt v1 (hash khóa trong registry và trong `03_configs/frozen/`), snapshot `gpt-4o-mini-2024-07-18`,
temperature 0. Notebook không in chỉ số nào và không in nhãn của các tập khóa; việc chấm điểm nằm ở notebook 12.

**Phạm vi gọi API**

Tầng 1: toàn bộ Hold-out (1.000) và Lazada (500). Tầng 2: hợp của các review gold-actionable và các review được tầng 1 của bất kỳ
chiến lược nào (seed 42) gọi ACTIONABLE — đủ cho đánh giá tầng 2, cascade và hybrid 3×3. Focused set: chỉ tầng 2 trên các review
gold-actionable. Nhãn gold chỉ được dùng để chọn phạm vi gọi, không được in ra.

In [1]:
import os, sys, json, datetime
import numpy as np, pandas as pd
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification.final_eval import LOCKED, TEXT_ONLY, export_text_only, load_frozen
DATA = os.path.join(ROOT, '01_data', 'processed')
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
FZ = load_frozen(ROOT)
print('Prompt v1 — hash tầng 1:', FZ['tier1']['llm']['prompt_hash'][:16], '| tầng 2:', FZ['tier2']['llm']['prompt_hash'][:16])

Prompt v1 — hash tầng 1: 9730222c472ff4a7 | tầng 2: 973a305b8e73efbb


## Phần A — xuất file văn bản cho Kaggle (chạy trước 10a)

**Nội dung file**

Chỉ hai cột `review_id` và `text`. Ba file `holdout_text_only.csv`, `focused_text_only.csv`, `lazada_text_only.csv` được ghi vào
`01_data/processed/` (thư mục không đưa lên Git) để tải lên dataset Kaggle `kltn-shopee-gold` cùng Train và Validation.

In [2]:
counts = export_text_only(DATA)
print('Đã xuất (số review):', counts)
for name in LOCKED:
    cols = list(pd.read_csv(os.path.join(DATA, TEXT_ONLY.format(name)), nrows=1, encoding='utf-8-sig').columns)
    assert cols == ['review_id', 'text'], cols
print('Các file chỉ có review_id và text.')

Đã xuất (số review): {'holdout': 1000, 'focused': 150, 'lazada': 500}
Các file chỉ có review_id và text.


## Phần B — gọi LLM (chạy sau 10a)

**Điều kiện**

Cần các file `gd10_pred_holdout.csv`, `gd10_pred_focused.csv`, `gd10_pred_lazada.csv` của notebook 10 trong
`05_experiments/rq1_classification/`. Nếu chưa có, phần B dừng tại đây.

In [3]:
PRED = {s: os.path.join(RES, f'gd10_pred_{s}.csv') for s in LOCKED}
READY = all(os.path.exists(p) for p in PRED.values())
print('Đã có dự đoán của 10a:', READY)
if READY:
    pred = {s: pd.read_csv(p) for s, p in PRED.items()}
    texts = {s: pd.read_csv(os.path.join(DATA, f), usecols=['review_id', 'text'], encoding='utf-8-sig') for s, f in LOCKED.items()}
    gold_act = {s: set(pd.read_csv(os.path.join(DATA, f), usecols=['review_id', 'gold_label'], encoding='utf-8-sig')
                       .query("gold_label != 'OTHER'")['review_id']) for s, f in LOCKED.items()}   # chỉ để chọn phạm vi gọi

    def neural_act(s, model):
        d = pred[s][(pred[s]['tier'] == 'tier1') & (pred[s]['model'] == model) & (pred[s]['seed'] == 42)]
        return set(d.loc[d['p_ACTIONABLE'] >= FZ['tier1'][model]['threshold'], 'review_id'])

Đã có dự đoán của 10a: True


**Kiểm tra API và prompt**

Gọi API một lần để chắc snapshot còn dùng được; nạp prompt v1 và kiểm tra hash trùng bản đã khóa (khác thì dừng).

In [4]:
if READY:
    from src.utils.llm import check_api, get_client, prices
    from src.classification.strategy_llm_fewshot.prompts import load_prompt, register
    from src.classification.strategy_llm_fewshot.classify import classify, cost_summary
    r = check_api()
    assert r['ok'] and r['snapshot_match'], r
    client, SETTINGS = get_client()
    PRICES = prices()
    train_df = pd.read_csv(os.path.join(DATA, 'shopee_train.csv'), encoding='utf-8-sig')
    PROMPTS = {t: load_prompt(t, 'v1', train_df) for t in (1, 2)}
    for t in (1, 2):
        h, status = register(PROMPTS[t], SETTINGS, datetime.date.today().strftime('%d/%m/%Y'))
        assert status == 'khớp bản đã khóa' and h == FZ[f'tier{t}']['llm']['prompt_hash'], (t, status)
    print('Model:', r['response_model'], '| prompt v1 khớp bản đã khóa | giá:', PRICES)

Model: gpt-4o-mini-2024-07-18 | prompt v1 khớp bản đã khóa | giá: {'input': 0.15, 'output': 0.6, 'source': 'env'}


## Chạy thử

**Mẫu**

5 review Hold-out cho tầng 1 và 5 review Hold-out cho tầng 2 (rút với seed 42), để kiểm tra cách parse và chi phí thật. Cell in nhãn do
LLM dự đoán, không in nhãn gold. Ước lượng chi phí của lần chạy đầy đủ được tính từ chi phí thật của mẫu này.

In [5]:
if READY:
    PILOT_LOG = os.path.join(RES, 'gd10_llm_pilot_calls.jsonl')
    sample = texts['holdout'].sample(10, random_state=42)
    p1 = classify(sample.iloc[:5], PROMPTS[1], client, SETTINGS, PILOT_LOG)
    p2 = classify(sample.iloc[5:], PROMPTS[2], client, SETTINGS, PILOT_LOG)
    print(pd.concat([p1, p2])[['review_id', 'tier', 'pred_label', 'attempts', 'fallback', 'prompt_tokens', 'completion_tokens']]
          .assign(review_id=lambda d: d['review_id'].str[:8]).to_string(index=False))
    usd = {t: cost_summary(p, PRICES, f'tầng {t}')['usd_/1000_review'] for t, p in ((1, p1), (2, p2))}
    union_est = {s: len(gold_act[s] | neural_act(s, 'visobert') | neural_act(s, 'bilstm')) for s in ('holdout', 'lazada')}
    n1 = len(texts['holdout']) + len(texts['lazada'])
    n2 = sum(union_est.values()) + len(gold_act['focused'])
    print(f'Ước lượng lần chạy đầy đủ: tầng 1 {n1} lượt gọi, tầng 2 khoảng {n2} lượt (chưa tính phần LLM tầng 1 gọi thêm) -> '
          f"~${(n1 * usd[1] + n2 * usd[2]) / 1000:.3f} (cận trên)")

review_id  tier     pred_label  attempts  fallback  prompt_tokens  completion_tokens
 389be78e     1     ACTIONABLE         1     False           1160                  6
 8d72db89     1 NOT_ACTIONABLE         1     False           1120                  7
 8e234a0f     1 NOT_ACTIONABLE         1     False           1113                  7
 6f758f01     1 NOT_ACTIONABLE         1     False           1115                  7
 05066429     1 NOT_ACTIONABLE         1     False           1115                  7
 76c2d680     2             UX         1     False           1032                  5
 61229734     2        FEATURE         1     False           1033                  5
 3442dc01     2             UX         1     False           1044                  5
 c0c857b7     2            BUG         1     False           1035                  5
 5cf51afb     2             UX         1     False           1027                  5
Ước lượng lần chạy đầy đủ: tầng 1 1500 lượt gọi, tầng 2 khoảng 66

## Chạy đầy đủ

**Điều kiện**

Chỉ chạy khi `CONFIRM_FULL_RUN = True`, sau khi kết quả chạy thử đã được xem và xác nhận. Nếu file dự đoán đã tồn tại thì không gọi
API lại.

**Thứ tự**

Tầng 1 trên Hold-out và Lazada → xác định phạm vi tầng 2 (gold-actionable ∪ tầng 1 của ba chiến lược gọi ACTIONABLE) → tầng 2 trên
Hold-out, Lazada và phần gold-actionable của focused set.

In [8]:
CONFIRM_FULL_RUN = True
OUTF = {(s, t): os.path.join(RES, f'gd10_llm_{s}_tier{t}.csv') for s in LOCKED for t in (1, 2)}
LOG = os.path.join(RES, 'gd10_llm_calls.jsonl')
if not READY:
    print('Chưa có dự đoán của 10a.')
elif not CONFIRM_FULL_RUN:
    print('Chưa chạy đầy đủ (CONFIRM_FULL_RUN = False).')
elif os.path.exists(OUTF[('holdout', 2)]):
    print('Đã có dự đoán LLM — không gọi API lại.')
else:
    r1 = {}
    for s in ('holdout', 'lazada'):
        r1[s] = classify(texts[s], PROMPTS[1], client, SETTINGS, LOG)
        r1[s].to_csv(OUTF[(s, 1)], index=False)
    scope = {s: gold_act[s] | neural_act(s, 'visobert') | neural_act(s, 'bilstm')
                | set(r1[s].loc[r1[s]['pred_label'] == 'ACTIONABLE', 'review_id']) for s in ('holdout', 'lazada')}
    scope['focused'] = gold_act['focused']
    for s in ('holdout', 'lazada', 'focused'):
        r2 = classify(texts[s][texts[s]['review_id'].isin(scope[s])], PROMPTS[2], client, SETTINGS, LOG)
        r2.to_csv(OUTF[(s, 2)], index=False)
    print('Đã lưu dự đoán LLM:', sorted(os.path.basename(p) for p in OUTF.values() if os.path.exists(p)))

[tier1] 50/1000
[tier1] 100/1000
[tier1] 150/1000
[tier1] 200/1000
[tier1] 250/1000
[tier1] 300/1000
[tier1] 350/1000
[tier1] 400/1000
[tier1] 450/1000
[tier1] 500/1000
[tier1] 550/1000
[tier1] 600/1000
[tier1] 650/1000
[tier1] 700/1000
[tier1] 750/1000
[tier1] 800/1000
[tier1] 850/1000
[tier1] 900/1000
[tier1] 950/1000
[tier1] 1000/1000
[tier1] 50/500
[tier1] 100/500
[tier1] 150/500
[tier1] 200/500
[tier1] 250/500
[tier1] 300/500
[tier1] 350/500
[tier1] 400/500
[tier1] 450/500
[tier1] 500/500
[tier2] 50/378
[tier2] 100/378
[tier2] 150/378
[tier2] 200/378
[tier2] 250/378
[tier2] 300/378
[tier2] 350/378
[tier2] 50/226
[tier2] 100/226
[tier2] 150/226
[tier2] 200/226
[tier2] 50/91
Đã lưu dự đoán LLM: ['gd10_llm_focused_tier2.csv', 'gd10_llm_holdout_tier1.csv', 'gd10_llm_holdout_tier2.csv', 'gd10_llm_lazada_tier1.csv', 'gd10_llm_lazada_tier2.csv']


**Chi phí thật của lần chạy**

Tổng hợp token, chi phí và thời gian từ dự đoán đã lưu; số `system_fingerprint` khác nhau được ghi lại như ở Giai đoạn 8.

In [9]:
if READY and os.path.exists(OUTF[('holdout', 2)]):
    rows = []
    for (s, t), p in OUTF.items():
        if os.path.exists(p):
            rows.append(cost_summary(pd.read_csv(p), PRICES, f'{s} – tầng {t}'))
    cost = pd.DataFrame(rows)
    cost.to_csv(os.path.join(RES, 'gd10_llm_costs.csv'), index=False)
    print(cost[['bước', 'n_review', 'n_lần_gọi', 'thử_lại_do_parse', 'fallback', 'token_vào', 'token_ra', 'chi_phí_usd',
                'giây_/1000_review']].round(4).to_string(index=False))
    calls = [json.loads(l) for l in open(LOG, encoding='utf-8')]
    print('Tổng: $%.4f | model trả về: %s | số system_fingerprint: %d' % (cost['chi_phí_usd'].sum(),
          sorted({c['model'] for c in calls}), len({c['system_fingerprint'] for c in calls})))

            bước  n_review  n_lần_gọi  thử_lại_do_parse  fallback  token_vào  token_ra  chi_phí_usd  giây_/1000_review
holdout – tầng 1      1000       1000                 0         0    1127930      6794       0.1733           913.0874
holdout – tầng 2       378        378                 0         0     397964      1890       0.0608           924.1137
focused – tầng 2        91         91                 0         0      97609       455       0.0149           908.0571
 lazada – tầng 1       500        500                 0         0     568698      3372       0.0873          1258.7609
 lazada – tầng 2       226        226                 0         0     240391      1130       0.0367           911.2148
Tổng: $0.3731 | model trả về: ['gpt-4o-mini-2024-07-18'] | số system_fingerprint: 6
